In [11]:
text = "hi 😉"
ids = list(text.encode("utf-8"))
# [104, 105, 32, 240, 159, 152, 137]   the emoji is 4 bytes


In [12]:
def get_stats(ids, counts=None):
    """get_stats([1, 2, 3, 1, 2]) -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}"""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    """merge([1, 2, 3, 1, 2], (1, 2), 4) -> [4, 3, 4]"""
    new_ids, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


In [13]:
def train(self, text, vocab_size):
    num_merges = vocab_size - 256
    ids = list(text.encode("utf-8"))
    merges = {}                                   # (id, id) -> new id
    vocab = {i: bytes([i]) for i in range(256)}   # id -> bytes
    for i in range(num_merges):
        stats = get_stats(ids)
        pair = max(stats, key=stats.get)          # most frequent pair
        idx = 256 + i                             # new token id
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    self.merges, self.vocab = merges, vocab

In [14]:
def decode(self, ids):
    text_bytes = b"".join(self.vocab[idx] for idx in ids)
    return text_bytes.decode("utf-8", errors="replace")

In [15]:
def encode(self, text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # the pair that was learned earliest (lowest new id)
        pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
        if pair not in self.merges:
            break                     # nothing left to merge
        ids = merge(ids, pair, self.merges[pair])
    return ids

In [16]:
import regex as re

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""


In [17]:
def bpe(mergeable_ranks, token, max_rank):
    parts = [bytes([b]) for b in token]
    while True:
        min_idx, min_rank = None, None
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx, min_rank = i, rank
        if min_rank is None or min_rank >= max_rank:
            break
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2:]
    return parts

def recover_merges(mergeable_ranks):
    merges = {}
    for token, rank in mergeable_ranks.items():
        if len(token) == 1:
            continue                                   # raw bytes aren't merges
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))
        assert len(pair) == 2
        merges[(mergeable_ranks[pair[0]], mergeable_ranks[pair[1]])] = rank
    return merges

In [18]:
import tiktoken

GPT4_SPECIAL_TOKENS = {
    "<|endoftext|>": 100257,
    "<|fim_prefix|>": 100258,
    "<|fim_middle|>": 100259,
    "<|fim_suffix|>": 100260,
    "<|endofprompt|>": 100276,
}

class RegexTokenizer:
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        self.pattern = pattern
        self.compiled_pattern = re.compile(pattern)
        self.merges = {}                  # (id, id) -> new id
        self.special_tokens = {}          # str -> id
        self.inverse_special_tokens = {}  # id -> str
        self.vocab = self._build_vocab()

    def _build_vocab(self):
        vocab = {i: bytes([i]) for i in range(256)}
        for (p0, p1), idx in sorted(self.merges.items(), key=lambda kv: kv[1]):
            vocab[idx] = vocab[p0] + vocab[p1]
        return vocab

    def register_special_tokens(self, special_tokens):
        self.special_tokens = special_tokens
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}

    def train(self, text, vocab_size):
        # same as before, but pairs never cross chunk boundaries
        chunks = [list(ch.encode("utf-8")) for ch in self.compiled_pattern.findall(text)]
        self.merges = {}
        for i in range(vocab_size - 256):
            stats = {}
            for ids in chunks:
                get_stats(ids, stats)
            pair = max(stats, key=stats.get)
            chunks = [merge(ids, pair, 256 + i) for ids in chunks]
            self.merges[pair] = 256 + i
        self.vocab = self._build_vocab()

    def decode(self, ids):
        parts = []
        for idx in ids:
            if idx in self.inverse_special_tokens:
                parts.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                parts.append(self.vocab[idx])
        return b"".join(parts).decode("utf-8", errors="replace")

    def _encode_chunk(self, text_bytes):
        ids = list(text_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def encode_ordinary(self, text):
        """encode, treating special tokens as plain text"""
        ids = []
        for chunk in self.compiled_pattern.findall(text):
            ids.extend(self._encode_chunk(chunk.encode("utf-8")))
        return ids

    def encode(self, text, allowed_special="none_raise"):
        """allowed_special: "all" | "none" | "none_raise" | a set of special token strings"""
        if allowed_special == "all":
            special = self.special_tokens
        elif allowed_special == "none":
            special = {}
        elif allowed_special == "none_raise":
            special = {}
            assert all(s not in text for s in self.special_tokens), "text contains special tokens"
        else:
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        if not special:
            return self.encode_ordinary(text)
        # split around special tokens, encode the ordinary pieces normally
        special_pattern = "(" + "|".join(re.escape(s) for s in special) + ")"
        ids = []
        for part in re.split(special_pattern, text):
            if part in special:
                ids.append(special[part])
            else:
                ids.extend(self.encode_ordinary(part))
        return ids

In [19]:
class GPT4Tokenizer(RegexTokenizer):
    def __init__(self):
        super().__init__(pattern=GPT4_SPLIT_PATTERN)
        ranks = tiktoken.get_encoding("cl100k_base")._mergeable_ranks
        self.merges = recover_merges(ranks)
        self.vocab = self._build_vocab()
        self.byte_shuffle = {b: ranks[bytes([b])] for b in range(256)}
        self.inverse_byte_shuffle = {v: k for k, v in self.byte_shuffle.items()}
        self.register_special_tokens(GPT4_SPECIAL_TOKENS)

    def _encode_chunk(self, text_bytes):
        text_bytes = bytes(self.byte_shuffle[b] for b in text_bytes)
        return super()._encode_chunk(text_bytes)

    def decode(self, ids):
        # special IDs -> their text; other IDs -> vocab bytes, un-shuffled
        ...

In [20]:
enc = tiktoken.get_encoding("cl100k_base")
tok = GPT4Tokenizer()
t = "<|endoftext|>Hello world this is one document<|endoftext|>And this is another<|endofprompt|>"

print("hello world :", tok.encode("hello world") == [15339, 1917])
print("encode text :", tok.encode(text) == enc.encode(text))
print("decode text :", tok.decode(enc.encode(text)) == text)
print("special     :", tok.encode(t, allowed_special="all") == enc.encode(t, allowed_special="all"))

hello world : True
encode text : True
decode text : False
special     : True
